In [ ]:
# Importar las librerías necesarias y las funciones
import pandas as pd

from src.utils import calculate_age, calculate_distance_km

In [ ]:
# Cargar los conjuntos de datos de entrenamiento y prueba
df_train = pd.read_csv('../data/fraudTrain.csv')
df_test = pd.read_csv('../data/fraudTest.csv')

# Confirmar que los datos se cargaron correctamente y mostrar sus dimensiones
print("Datasets loaded successfully!")
print(f"Train dataset shape: {df_train.shape}")
print(f"Test dataset shape: {df_test.shape}")

### FSelección de Características y Criterios de Eliminación

Para evitar el sobreajuste (overfitting) y optimizar el rendimiento del pipeline, se eliminaron variables no predictivas basándose en los siguientes criterios técnicos:

* **Identificadores Únicos y PII (Unnamed: 0, cc_num, first, last, street, trans_num):** Carecen de patrones de fraude reutilizables. Los números de tarjeta sintéticos (cc_num) no representan BINs reales, y las variables de identidad personal fuerzan al modelo a memorizar individuos en lugar de comportamientos generalizables.
* **Texto Libre de Alta Cardinalidad (merchant, job, city, state):** Contienen miles de valores de texto no estructurados. Las señales espaciales se capturan de forma limpia mediante la distancia Haversine (distance_km), mientras que el dominio comercial se preserva mediante la categoría.
* **Redundancia Temporal (unix_time):** Marca de tiempo numérica en bruto. Se reemplaza por características temporales estructuradas extraídas de trans_date_trans_time (por ejemplo, trans_hour, trans_day).

In [ ]:
# Definir la lista de columnas a eliminar en la primera etapa de preprocesamiento
cols_to_drop = [
    'Unnamed: 0', 'cc_num', 'first', 'last', 'street', 'trans_num', 'merchant', 'job', 'city', 'state', 'unix_time'
    ]

# Eliminar las columnas seleccionadas en los conjuntos de entrenamiento y prueba
# Usar errors='ignore' para evitar interrupciones si una columna ya fue eliminada
df_train = df_train.drop(columns=cols_to_drop, errors='ignore')
df_test = df_test.drop(columns=cols_to_drop, errors='ignore')

# Mostrar las dimensiones actualizadas después de eliminar las características
print(f"Train dataset shape: {df_train.shape}")
print(f"Test dataset shape: {df_test.shape}")

In [ ]:
# Convertir la columna 'trans_date_trans_time' de texto a formato datetime de pandas
df_train['trans_date_trans_time'] = pd.to_datetime(df_train['trans_date_trans_time'])
df_train['dob'] = pd.to_datetime(df_train['dob'])

# Aplicar la misma conversión al conjunto de prueba
df_test['trans_date_trans_time'] = pd.to_datetime(df_test['trans_date_trans_time'])
df_test['dob'] = pd.to_datetime(df_test['dob'])

In [ ]:
# Calcular la edad del cliente usando la función auxiliar 'calculate_age'
df_train["age"] = calculate_age(df_train)
df_test["age"] = calculate_age(df_test)

In [ ]:
# Extraer la hora de la transacción
df_train['trans_hour'] = df_train['trans_date_trans_time'].dt.hour
df_test['trans_hour'] = df_test['trans_date_trans_time'].dt.hour

# Extraer el día de la semana como valor numérico (0=Lunes, 6=Domingo)
df_train['trans_day'] = df_train['trans_date_trans_time'].dt.dayofweek
df_test['trans_day'] = df_test['trans_date_trans_time'].dt.dayofweek

In [ ]:
# Calcular la distancia (km) entre el cliente y el comercio usando 'calculate_distance_km'
df_train['distance_km'] = calculate_distance_km(df_train)
df_test['distance_km'] = calculate_distance_km(df_test)

In [ ]:
# Verificar las columnas resultantes y sus tipos de datos
print(f'Train: {df_train.columns}')
print(f'\nTest: {df_test.columns}')
print(f"\n{df_train['trans_day'].dtype}")
print(f"\n{df_test['trans_day'].dtype}")

In [ ]:
# Eliminar columnas cuya información útil ya fue extraída
df_train = df_train.drop(
    columns=[
        'trans_date_trans_time', 'dob', 'lat', 'long',
        'merch_lat', 'merch_long', 'zip', 'city_pop'
    ],
    errors='ignore')

df_test = df_test.drop(
    columns=[
        'trans_date_trans_time', 'dob', 'lat', 'long',
        'merch_lat', 'merch_long', 'zip', 'city_pop'
    ],
    errors='ignore')

# Mostrar el conjunto final de características
print(f'Train: {df_train.columns}')
print(f'\nTest: {df_test.columns}')

In [ ]:
# Codificar la característica binaria 'gender' (Femenino = 0, Masculino = 1)
df_train['gender'] = df_train['gender'].map({'F': 0, 'M': 1})
df_test['gender'] = df_test['gender'].map({'F': 0, 'M': 1})

# Aplicar One-Hot Encoding a la variable categórica 'category'
# Usar drop_first=True para evitar la multicolinealidad perfecta
df_train = pd.get_dummies(df_train, columns=['category'], drop_first=True)
df_test = pd.get_dummies(df_test, columns=['category'], drop_first=True)

In [ ]:
# Verificar que ambos conjuntos contengan exactamente las mismas características y en el mismo orden
print(list(df_train.columns) == list(df_test.columns))

In [ ]:
# Separar el conjunto de entrenamiento en variables predictoras (X) y variable objetivo (y)
X_train = df_train.drop(columns=['is_fraud'])
y_train = df_train['is_fraud']

# Separar el conjunto de prueba en variables predictoras (X) y variable objetivo (y)
X_test = df_test.drop(columns=['is_fraud'])
y_test = df_test['is_fraud']

In [ ]:
# Exportar los datos procesados en formato Parquet comprimido para optimizar lectura/escritura
X_train.to_parquet('../data/X_train.parquet', index=False)
X_test.to_parquet('../data/X_test.parquet', index=False)

y_train.to_frame().to_parquet('../data/y_train.parquet', index=False)
y_test.to_frame().to_parquet('../data/y_test.parquet', index=False)

In [ ]:
# Mostrar las dimensiones finales del dataset
print("Filas y columnas de Train:", df_train.shape)
print("Filas y columnas de Test:", df_test.shape)